<a href="https://colab.research.google.com/github/zlfaris/DataMiningFix/blob/main/TgsDataMining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv('/content/Tweets.csv')
df = df.drop_duplicates()

df = df.rename(columns={
    'Tweet': 'text',
    'tweet': 'text',
    'Text': 'text',
    'airline_sentiment': 'sentiment'
})


def clean_text(x):
    x = str(x).lower()
    x = re.sub(r"http\S+", "", x)
    x = re.sub(r"@\w+", "", x)
    x = re.sub(r"[^a-z ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["clean"] = df["text"].astype(str).apply(clean_text)


X = df["clean"]
y = df["sentiment"]

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=5, stratify=y
)

vectorizer = TfidfVectorizer(
    min_df=3,
    max_df=0.9,
    sublinear_tf=True
)

x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)

model_bnb = BernoulliNB()
model_bnb.fit(x_train_vec, y_train)

svm_base = LinearSVC(C=1.0, max_iter=20000)
model_svm = CalibratedClassifierCV(svm_base, cv=3)
model_svm.fit(x_train_vec, y_train)

ensemble_model = VotingClassifier(
    estimators=[
        ('bnb', BernoulliNB()),
        ('svm', CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=20000), cv=3)),
    ],
    voting='soft'
)
ensemble_model.fit(x_train_vec, y_train)

print("Akurasi Bernoulli NB :", accuracy_score(y_test, model_bnb.predict(x_test_vec)))
print("Akurasi Linear SVM   :", accuracy_score(y_test, model_svm.predict(x_test_vec)))
print("Akurasi Ensemble     :", accuracy_score(y_test, ensemble_model.predict(x_test_vec)))


joblib.dump(model_bnb, "model_bernoulli_nb.pkl")
joblib.dump(model_svm, "model_linear_svm.pkl")
joblib.dump(ensemble_model, "model_ensemble_voting.pkl")
joblib.dump(vectorizer, "vectorizer_tfidf.pkl")

print("Model berhasil disimpan.")

Akurasi Bernoulli NB : 0.7692571037315987
Akurasi Linear SVM   : 0.7829510441629579
Akurasi Ensemble     : 0.7832933926737419
Model berhasil disimpan.
